# Historical Analysis

Descriptive statistics by province and canton, per hazard type and combined: recurrence and persistence, impact statistics, ENSO-category frequency and temporal trend (Mann-Kendall with Sen's slope).

Outputs: `hist_*` tables in `data/processed/`.

## 0. Setup

In [ ]:

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Tabular data handling
import pandas as pd
# Numerical operations
import numpy as np
# Mann-Kendall trend test
import pymannkendall as mk
# Efficient reader for GeoPackage layers
import pyogrio

# Show up to 40 columns when printing DataFrames
pd.set_option("display.max_columns", 40)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step3_historical_analysis", config, log_dir=PROJECT_ROOT / "logs")
# Log that the notebook started and which config was used
logger.info("análisis histórico iniciada — configuración cargada desde %s", CONFIG_PATH)

# First year of the analysis window
YEAR_MIN = config["date_range"]["start_year"]
# Last year of the analysis window
YEAR_MAX = config["date_range"]["end_year"]
# List of all years in the analysis window
ALL_YEARS = list(range(YEAR_MIN, YEAR_MAX + 1))

## 1. Load data

In [ ]:
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Load the hazard events table without geometry
he = pyogrio.read_dataframe(processed_dir / "hazard_event_geo.gpkg", read_geometry=False)
# Log the number of events loaded
logger.info("hazard_event_geo cargado: %d filas", len(he))

# Load the official canton and province names
adm2 = pyogrio.read_dataframe(
    processed_dir / "admin_boundaries.gpkg", layer="adm2_canton", read_geometry=False
)[["canton_code", "canton_name_official", "province_name_official"]].drop_duplicates()
# Remember the row count to detect accidental changes
n_antes = len(he)
# Attach official canton and province names to each event
he = he.drop(columns=["province_name_official"]).merge(adm2, on="canton_code", how="left")
# Check the name join neither duplicates nor loses events
assert len(he) == n_antes, "el join adm2 no debe duplicar ni perder filas"
# Count events without an official canton or province name
n_sin_nombre = he[["canton_name_official", "province_name_official"]].isna().any(axis=1).sum()
# Warn only if some events lack a name
if n_sin_nombre:
    # Log a warning so the correction is auditable
    logger.warning(
        "%d eventos sin nombre oficial de cantón/provincia tras el join por "
        "canton_code (canton_code no presente en admin_boundaries adm2_canton)",
        n_sin_nombre,
    )
# Log the coverage check
logger.info(
    "canton_name_official/province_name_official incorporados por atributo "
    "(canton_code, 100%% de cobertura); sin match: %d", n_sin_nombre
)

# Impact variables to summarize
IMPACT_VARS = [
    "deaths", "affected_direct", "affected_indirect", "sheltered",
    "houses_affected", "houses_destroyed", "schools_affected",
    "health_facilities_affected", "crops_affected_ha", "livestock_affected",
]

## 2. Recurrence and temporal persistence by administrative unit

In [ ]:
# Recurrence and persistence indicators per administrative unit
def resumen_recurrencia(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Summarize recurrence and persistence per administrative unit.
    
    Args:
        df: Hazard events with year and month columns.
        group_cols: Columns that define the unit.
    
    Returns:
        Table with event count, years, months and days affected per unit.
    """
    # Group events by the requested administrative columns
    g = df.groupby(group_cols)
    # Count events and distinct years, months and days affected
    out = g.agg(
        n_eventos=("event_id", "count"),
        anios_afectados=("year", "nunique"),
        primer_evento=("event_date", "min"),
        ultimo_evento=("event_date", "max"),
    ).reset_index()
    # Count the distinct year-month combinations per unit
    meses = df.groupby(group_cols).apply(
        lambda d: d[["year", "month"]].drop_duplicates().shape[0], include_groups=False
    ).rename("meses_afectados").reset_index()
    # Attach the month count to the summary
    out = out.merge(meses, on=group_cols, how="left")
    # Return the per-unit table
    return out

## 3. Impact statistics (mean, median, maximum, standard deviation)

In [ ]:
# Impact statistics per unit and impact variable
def resumen_impactos(df: pd.DataFrame, group_cols: list[str], impact_vars: list[str]) -> pd.DataFrame:
    """Summarize impact variables (mean, median, maximum, standard deviation, count) per unit.
    
    Args:
        df: Hazard events.
        group_cols: Columns that define the unit.
        impact_vars: Impact columns to summarize.
    
    Returns:
        Long table with one row per unit and impact variable.
    """
    # List that collects one table per variable
    rows = []
    # Loop over the impact variables
    for var in impact_vars:
        # Mean, median, maximum, standard deviation and count of the variable per unit
        g = df.groupby(group_cols)[var].agg(
            n="count", media="mean", mediana="median", maximo="max", desv_std="std"
        ).reset_index()
        # Record which variable the table describes
        g["variable_impacto"] = var
        # Store the table
        rows.append(g)
    # Stack the per-variable tables
    result = pd.concat(rows, ignore_index=True)
    # A unit with one event has no standard deviation; set it to zero
    result["desv_std"] = result["desv_std"].fillna(0)
    # Return the final table
    return result

## 4. Frequency by ENSO category (ICEN and ONI separately)

In [ ]:
# Event counts by ENSO category for each index
def resumen_enso(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Count events by ENSO category, for ICEN and ONI separately.
    
    Args:
        df: Hazard events with ICEN and ONI category columns.
        group_cols: Columns that define the unit.
    
    Returns:
        Table with event counts per unit, index and category.
    """
    # Events per unit and ICEN category
    icen = df.groupby(group_cols + ["icen_category"]).size().rename("n_eventos").reset_index()
    # Standardize the category column name
    icen = icen.rename(columns={"icen_category": "categoria"})
    # Label the rows as ICEN
    icen["indice_enso"] = "ICEN"

    # Events per unit and ONI category
    oni = df.groupby(group_cols + ["oni_category"]).size().rename("n_eventos").reset_index()
    # Standardize the category column name
    oni = oni.rename(columns={"oni_category": "categoria"})
    # Label the rows as ONI
    oni["indice_enso"] = "ONI"

    # Stack both indices
    return pd.concat([icen, oni], ignore_index=True)

## 5. Temporal trend (Mann-Kendall and Sen's slope)

In [ ]:
# Annual-count trend per unit (Mann-Kendall test and Sen's slope)
def tendencia_mann_kendall(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Test the annual event-count trend per unit with Mann-Kendall and Sen's slope.
    
    Args:
        df: Hazard events with a year column.
        group_cols: Columns that define the unit.
    
    Returns:
        Table with trend direction, p-value, slope and years with data per unit.
    """
    # Minimum number of years with events to run the test
    MIN_ANIOS = 4
    # List that collects one row per unit
    results = []
    # Loop over each unit
    for keys, sub in df.groupby(group_cols):
        # Make the group key a tuple
        keys = keys if isinstance(keys, tuple) else (keys,)
        # Annual event counts, with zero for years without events
        annual = sub.groupby("year").size().reindex(ALL_YEARS, fill_value=0)
        # Number of years that have at least one event
        n_anios_con_dato = (annual > 0).sum()
        # Skip units with too few years
        if n_anios_con_dato < MIN_ANIOS:
            # Record 'insufficient data' for those units
            results.append((*keys, "datos_insuficientes", np.nan, np.nan, n_anios_con_dato))
            # Move on to the next unit
            continue
        # Run the Mann-Kendall test
        test = mk.original_test(annual.values)
        # Record the trend, p-value, Sen's slope and number of years
        results.append((*keys, test.trend, round(test.p, 4), round(test.slope, 4), n_anios_con_dato))
    # Column names of the result
    cols = group_cols + ["tendencia", "p_valor", "pendiente_sen", "anios_con_dato"]
    # Return the trend table
    return pd.DataFrame(results, columns=cols)

## 6. Run: canton and province level, per hazard and combined

In [ ]:
# Compute all indicator tables for one administrative level
def ejecutar_nivel(df: pd.DataFrame, admin_cols: list[str]) -> dict[str, pd.DataFrame]:
    """Compute recurrence, impact, ENSO and trend tables for one administrative level.
    
    Args:
        df: Hazard events with official names.
        admin_cols: Columns that define the administrative level.
    
    Returns:
        Dictionary of tables, each stacking the per-hazard and the combined views.
    """
    # Grouping columns for the per-hazard view
    group_por_hazard = admin_cols + ["hazard_type"]

    # Recurrence per hazard
    recurrencia_h = resumen_recurrencia(df, group_por_hazard)
    # Impact statistics per hazard
    impactos_h = resumen_impactos(df, group_por_hazard, IMPACT_VARS)
    # ENSO frequency per hazard
    enso_h = resumen_enso(df, group_por_hazard)
    # Trend per hazard
    tendencia_h = tendencia_mann_kendall(df, group_por_hazard)

    # Copy the events for the combined view
    df_todos = df.copy()
    # Label every event as 'TODOS' (all hazards)
    df_todos["hazard_type"] = "TODOS"
    # Recurrence for all hazards combined
    recurrencia_t = resumen_recurrencia(df_todos, group_por_hazard)
    # Impact statistics for all hazards combined
    impactos_t = resumen_impactos(df_todos, group_por_hazard, IMPACT_VARS)
    # ENSO frequency for all hazards combined
    enso_t = resumen_enso(df_todos, group_por_hazard)
    # Trend for all hazards combined
    tendencia_t = tendencia_mann_kendall(df_todos, group_por_hazard)

    # Return all tables, stacking per-hazard and combined views
    return {
        "recurrencia": pd.concat([recurrencia_h, recurrencia_t], ignore_index=True),
        "impactos": pd.concat([impactos_h, impactos_t], ignore_index=True),
        "enso": pd.concat([enso_h, enso_t], ignore_index=True),
        "tendencia": pd.concat([tendencia_h, tendencia_t], ignore_index=True),
    }

# Columns that identify a canton
CANTON_COLS = ["province_code", "province_name_official", "canton_code", "canton_name_official"]
# Columns that identify a province
PROVINCE_COLS = ["province_code", "province_name_official"]

# Log the start of the canton-level run
logger.info("Calculando indicadores a nivel cantón...")
# Compute all indicators at canton level
resultados_canton = ejecutar_nivel(he, CANTON_COLS)

# Log the start of the province-level run
logger.info("Calculando indicadores a nivel provincia...")
# Compute all indicators at province level
resultados_provincia = ejecutar_nivel(he, PROVINCE_COLS)

# Loop over the canton tables
for nombre, tabla in resultados_canton.items():
    # Log each canton table size
    logger.info("Cantón — tabla '%s': %d filas", nombre, len(tabla))
# Loop over the province tables
for nombre, tabla in resultados_provincia.items():
    # Log each province table size
    logger.info("Provincia — tabla '%s': %d filas", nombre, len(tabla))

## 7. Validation

In [ ]:
# Total events counted at canton level in the combined view
total_todos_canton = resultados_canton["recurrencia"].loc[
    resultados_canton["recurrencia"]["hazard_type"] == "TODOS", "n_eventos"
].sum()
# Log the total against the expected number of events
logger.info("Suma n_eventos (vista TODOS, nivel cantón): %d (esperado %d)", total_todos_canton, len(he))
# Check the canton aggregation matches the total number of events
assert total_todos_canton == len(he), "La agregación por cantón no cuadra con el total de eventos"

# Total events counted at province level in the combined view
total_todos_provincia = resultados_provincia["recurrencia"].loc[
    resultados_provincia["recurrencia"]["hazard_type"] == "TODOS", "n_eventos"
].sum()
# Check the province aggregation matches the total number of events
assert total_todos_provincia == len(he), "La agregación por provincia no cuadra con el total de eventos"

# Number of cantons with events
n_cantones_con_eventos = he.groupby(CANTON_COLS).ngroups
# Number of cantons in the trend table
n_cantones_en_tendencia = resultados_canton["tendencia"]["canton_code"].nunique()
# Log both numbers
logger.info("Cantones con eventos: %d | cantones en tabla de tendencia: %d", n_cantones_con_eventos, n_cantones_en_tendencia)
# Check no canton is missing from the trend table
assert n_cantones_con_eventos == n_cantones_en_tendencia

# Ten cantons with the most events
top10 = resultados_canton["recurrencia"][resultados_canton["recurrencia"]["hazard_type"]=="TODOS"] \
    .sort_values("n_eventos", ascending=False).head(10)
# Print the top ten
print(top10[["province_name_official","canton_name_official","n_eventos","anios_afectados","meses_afectados"]].to_string(index=False))

# Print a header for the trend distribution
print("\nDistribución de tendencias (cantón, TODOS los peligros):")
# Print how many cantons fall in each trend class
print(resultados_canton["tendencia"].loc[resultados_canton["tendencia"]["hazard_type"]=="TODOS", "tendencia"].value_counts())

## 8. Export

In [ ]:
# Create the output folder if needed
out_dir = ensure_dir(processed_dir)
# List of exported table names
written = []
# Loop over the two administrative levels
for nivel, resultados in [("canton", resultados_canton), ("provincia", resultados_provincia)]:
    # Loop over the tables of each level
    for nombre_tabla, df_tabla in resultados.items():
        # File base name
        base_name = f"hist_{nombre_tabla}_{nivel}"
        # Parquet path
        parquet_path = out_dir / f"{base_name}.parquet"
        # CSV path
        csv_path = out_dir / f"{base_name}.csv"
        # Write Parquet
        df_tabla.to_parquet(parquet_path, index=False)
        # Write CSV
        df_tabla.to_csv(csv_path, index=False, encoding="utf-8")
        # Remember the exported name
        written.append(base_name)
        # Log each exported table
        logger.info("Exportado: %s (%d filas)", base_name, len(df_tabla))

# Print the output folder
print("Tablas exportadas a", out_dir, ":")
# Loop over the exported names
for w in written:
    # Print each name
    print(" -", w)